In [1]:
import gzip
import numpy as np
import pandas as pd

In [2]:
afr_vcf = '../data/1000G/HLA_typed/AFR.qc.no_monomorphic.vcf.gz'
amr_vcf = '../data/1000G/HLA_typed/AMR.qc.no_monomorphic.vcf.gz'
eas_vcf = '../data/1000G/HLA_typed/EAS.qc.no_monomorphic.vcf.gz'
eur_vcf = '../data/1000G/HLA_typed/EUR.qc.no_monomorphic.vcf.gz'
sas_vcf = '../data/1000G/HLA_typed/SAS.qc.no_monomorphic.vcf.gz'

all_noafr_vcf = '../data/1000G/HLA_typed/ALL_noAFR.qc.no_monomorphic.vcf.gz'
all_noamr_vcf = '../data/1000G/HLA_typed/ALL_noAMR.qc.no_monomorphic.vcf.gz'
all_noeas_vcf = '../data/1000G/HLA_typed/ALL_noEAS.qc.no_monomorphic.vcf.gz'
all_noeur_vcf = '../data/1000G/HLA_typed/ALL_noEUR.qc.no_monomorphic.vcf.gz'
all_nosas_vcf = '../data/1000G/HLA_typed/ALL_noSAS.qc.no_monomorphic.vcf.gz'

all_vcf = '../data/1000G/HLA_typed/ALL.qc.no_monomorphic.vcf.gz'

datasets = {
    'ALL': all_vcf,
    'AFR': afr_vcf,
    'AMR': amr_vcf,
    'EAS': eas_vcf,
    'EUR': eur_vcf,
    'SAS': sas_vcf,
    'ALL_noAFR': all_noafr_vcf,
    'ALL_noAMR': all_noamr_vcf,
    'ALL_noEAS': all_noeas_vcf,
    'ALL_noEUR': all_noeur_vcf,
    'ALL_noSAS': all_nosas_vcf,
}

In [3]:
# Load HLA allele matrix
def load_hla(vcf, mac_threshold):
    header = None
    samples = None
    
    with gzip.open(vcf, 'rt') as ifile:
        for line in ifile:
            if line.startswith("#CHROM"):
                header = line.rstrip().split('\t')
                assert len(header) > 9
                samples = header[9:]
                break
    
    assert header is not None
    assert samples is not None and len(samples) > 0
    
    rows = []
    for sample in samples:
        rows.append(dict())
    
    with gzip.open(vcf, 'rt') as ifile:
        for line in ifile:
            if line.startswith('#'): continue
            fields = dict(zip(header, line.rstrip().split('\t')))
            allele = fields['ID'].removeprefix('HLA-')
    
            # Keep only classical Class I and Class II
            if allele.split('*')[0] not in {'A', 'B', 'C', 'DPA1', 'DPB1', 'DQB1', 'DRA', 'DRB1'}: continue
            
            format_field = fields['FORMAT'].split(':')
            for i, sample in enumerate(samples):
                sample_field = dict(zip(format_field, fields[sample].split(':')))
                gt = sample_field['GT']
                if gt == './.':
                    gt = np.nan
                else:
                    gt = sum(float(x) for x in gt.split('/'))
                rows[i][allele] = gt
            
    hla_df = pd.DataFrame(rows)
    
    # check if no monomorphic
    assert all(hla_df.var() > 0) 
    
    # Remove alleles by MAC
    mac = hla_df.sum(axis=0)
    hla_df = hla_df.loc[:, mac >= mac_threshold]
    return hla_df

# correlation matrix
def comp_corr(df):
    corr = df.corr(method = "pearson")
    # correlation matrix should have no NAs
    assert corr.isna().sum().sum() == 0
    return corr

# Compute Eigenvalues
def eigenvalues(corr):
    eigvals = np.linalg.eigvalsh(corr)
    eigvals = np.sort(eigvals)[::-1]
    return sum(eigvals > 1)

# Compute effective tests
def compute_meff(corr):
    eigvals = np.linalg.eigvalsh(corr)
    eigvals = np.sort(eigvals)[::-1]
    li_ji_contrib = np.minimum(eigvals, 1)
    cum_meff = np.cumsum(li_ji_contrib)
    final_meff = cum_meff[-1]
    return final_meff

In [4]:
mac = 5
print(f'MAC >= {mac}')
print('Dataset\tN_PCs\tN_PCs_with_lamda>1\tM_eff\tM_eff Difference')
meff_baseline = None
for label, dataset_vcf in datasets.items():
    df = load_hla(dataset_vcf, mac)
    corr = comp_corr(df)
    n_pcs = eigenvalues(corr)
    meff = compute_meff(corr)
    if (label == 'ALL'):
        meff_baseline = meff
    diff_meff = meff - meff_baseline
    
    print(f'{label}\t{len(corr.columns)}\t{n_pcs}\t{meff:.0f}\t{diff_meff}')

MAC >= 5
Dataset	N_PCs	N_PCs_with_lamda>1	M_eff	M_eff Difference
ALL	283	116	202	0.0
AFR	148	58	98	-104.57292057009913
AMR	153	57	91	-111.51079928638187
EAS	135	48	79	-122.67256772939015
EUR	133	51	82	-120.58297757525062
SAS	138	53	82	-119.74976092188575
ALL_noAFR	245	99	168	-33.77019230798524
ALL_noAMR	261	106	184	-18.567694940419443
ALL_noEAS	256	105	179	-23.328785996623623
ALL_noEUR	270	110	190	-12.150891021923456
ALL_noSAS	264	106	185	-17.13825807864012


In [5]:
mac_thresholds = [5, 10, 20, 50]

print(f'Dataset;', end = "")
for mac in mac_thresholds:
    print(f';N_PCs (MAC>={mac});M_eff (MAC>={mac})', end = "")
print()

for label in ['ALL', 'AFR', 'AMR', 'EAS', 'EUR', 'SAS']:
    print(f'{label};', end = "")
    for mac in mac_thresholds:
        df = load_hla(datasets[label], mac)
        corr = comp_corr(df)
        meff = compute_meff(corr)
        print(f';{len(corr.columns)};{meff:.0f} ({meff / len(corr.columns) * 100:.0f})', end = "")
    print()
    

Dataset;;N_PCs (MAC>=5);M_eff (MAC>=5);N_PCs (MAC>=10);M_eff (MAC>=10);N_PCs (MAC>=20);M_eff (MAC>=20);N_PCs (MAC>=50);M_eff (MAC>=50)
ALL;;283;202 (71);236;165 (70);196;135 (69);143;98 (69)
AFR;;148;98 (66);128;84 (65);103;68 (66);62;40 (65)
AMR;;153;91 (59);109;67 (61);64;40 (63);21;15 (70)
EAS;;135;79 (59);110;64 (58);84;50 (60);44;28 (63)
EUR;;133;82 (61);105;64 (61);77;47 (61);43;26 (61)
SAS;;138;82 (60);107;66 (61);77;48 (62);46;29 (62)
